# Trace Count v4: v2-Style Steering

This notebook follows `pipeline_v4_steering_codex_prompt.md`. It intentionally reuses the v2 symbolic counting setup and v2-style HuggingFace GPT-2 architecture with learned absolute positional embeddings. The new part is mechanistic: hidden-state cache, probes, direction extraction, steering, and patching.

In [8]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
if Path('/content').exists():
    repo_dir = Path('/content/Synthetic_CoT_NiaH_Count')
    if repo_dir.exists():
        os.chdir(repo_dir)
        subprocess.run(['git', 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
        os.chdir(repo_dir)

def patch_v4_model_hook_for_new_transformers():
    path = Path('synthetic_niah_v4/model.py')
    if not path.exists():
        print('v4 model hook patch skipped; file not found:', path)
        return
    text = path.read_text(encoding='utf-8')
    old = '''                handles.append(
                    block.register_forward_pre_hook(
                        lambda _module, inputs, layer_idx=layer_idx: (
                            apply(f"resid_pre_layer_{layer_idx}", inputs[0]),
                            *inputs[1:],
                        )
                    )
                )

                def post_hook(_module, _inputs, output, layer_idx=layer_idx):
                    hidden = apply(f"resid_post_layer_{layer_idx}", output[0])
                    return (hidden, *output[1:])
'''
    new = '''                def pre_hook(_module, inputs, layer_idx=layer_idx):
                    if not inputs:
                        return inputs
                    hidden = inputs[0]
                    if isinstance(hidden, tuple):
                        if not hidden:
                            return inputs
                        patched = (apply(f"resid_pre_layer_{layer_idx}", hidden[0]), *hidden[1:])
                        return (patched, *inputs[1:])
                    return (apply(f"resid_pre_layer_{layer_idx}", hidden), *inputs[1:])

                handles.append(
                    block.register_forward_pre_hook(pre_hook)
                )

                def post_hook(_module, _inputs, output, layer_idx=layer_idx):
                    if isinstance(output, tuple):
                        hidden = apply(f"resid_post_layer_{layer_idx}", output[0])
                        return (hidden, *output[1:])
                    return apply(f"resid_post_layer_{layer_idx}", output)
'''
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        print('Patched synthetic_niah_v4/model.py for newer Transformers/PyTorch hook outputs.')
    elif 'return apply(f"resid_post_layer_{layer_idx}", output)' in text:
        print('v4 model hook patch already present.')
    else:
        print('v4 model hook patch not applied; source pattern did not match.')

patch_v4_model_hook_for_new_transformers()

def patch_v4_steering_projection_r2_merge():
    path = Path('synthetic_niah_v4/steering.py')
    if not path.exists():
        print('v4 steering patch skipped; file not found:', path)
        return
    text = path.read_text(encoding='utf-8')
    changed = False
    if 'if "projection_r2" not in direction_df.columns:' not in text:
        text = text.replace(
            '    if direction_df.empty:\n        return direction_df\n    sub = direction_df[\n',
            '    if direction_df.empty:\n        return direction_df\n    if "projection_r2" not in direction_df.columns:\n        direction_df = direction_df.copy()\n        direction_df["projection_r2"] = 0.0\n    sub = direction_df[\n',
        )
        changed = True
    if 'def _load_direction_configs(run_dir: Path) -> pd.DataFrame:' not in text:
        helper = '''\n\ndef _load_direction_configs(run_dir: Path) -> pd.DataFrame:\n    artifacts = run_dir / "artifacts"\n    tables = run_dir / "tables"\n    direction_meta_path = artifacts / "directions.csv"\n    direction_metrics_path = tables / "direction_metrics.csv"\n    direction_df = pd.read_csv(direction_meta_path) if direction_meta_path.exists() else pd.DataFrame()\n    if direction_df.empty:\n        return direction_df\n    metrics_df = pd.read_csv(direction_metrics_path) if direction_metrics_path.exists() and direction_metrics_path.stat().st_size > 0 else pd.DataFrame()\n    if metrics_df.empty or "projection_r2" not in metrics_df.columns:\n        direction_df = direction_df.copy()\n        direction_df["projection_r2"] = 0.0\n        direction_df["projection_slope"] = 0.0\n        return direction_df\n    key_cols = ["model_type", "eval_mode", "anchor_name", "anchor_k", "hook_name", "layer", "direction_type", "target"]\n    left = direction_df.copy()\n    right = metrics_df.copy()\n    for col in key_cols:\n        if col not in left.columns:\n            left[col] = ""\n        if col not in right.columns:\n            right[col] = ""\n        left[f"__key_{col}"] = left[col].fillna("").astype(str)\n        right[f"__key_{col}"] = right[col].fillna("").astype(str)\n    merge_cols = [f"__key_{col}" for col in key_cols]\n    keep = merge_cols + ["projection_r2", "projection_slope"]\n    merged = left.merge(right[keep].drop_duplicates(merge_cols), on=merge_cols, how="left")\n    merged["projection_r2"] = merged["projection_r2"].fillna(0.0)\n    merged["projection_slope"] = merged["projection_slope"].fillna(0.0)\n    return merged.drop(columns=merge_cols)\n'''
        text = text.replace('\n\ndef _distribution(preds: list[int]) -> np.ndarray:\n', helper + '\n\ndef _distribution(preds: list[int]) -> np.ndarray:\n')
        changed = True
    old = '    direction_meta_path = artifacts / "directions.csv"\n    direction_npz_path = artifacts / "directions.npz"\n    direction_df = pd.read_csv(direction_meta_path) if direction_meta_path.exists() else pd.DataFrame()\n'
    new = '    direction_npz_path = artifacts / "directions.npz"\n    direction_df = _load_direction_configs(run_dir)\n'
    if old in text:
        text = text.replace(old, new)
        changed = True
    if changed:
        path.write_text(text, encoding='utf-8')
        print('Patched synthetic_niah_v4/steering.py to merge projection_r2 from direction_metrics.csv.')
    else:
        print('v4 steering projection_r2 patch already present.')

patch_v4_steering_projection_r2_merge()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('cwd =', Path.cwd())

v4 model hook patch already present.
v4 steering projection_r2 patch already present.
cwd = /content/Synthetic_CoT_NiaH_Count


## Runtime settings

- `debug`: quick end-to-end sanity check.
- `main`: v2-style main setting: two models, one seed by default, `1..10` counts, GPT-2 learned absolute positions.
- `stage='all'` runs training, behavior eval, hidden cache, probes, direction extraction, patching, steering, plots, and a markdown/html artifact report.

In [9]:
PRESET = 'debug'  # 'debug' or 'main'
STAGE = 'all'
SEEDS = '1234'
OUT_ROOT = 'runs/synthetic_niah_v4'
RUN_NAME = ''
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
SKIP_COMPLETED = True

print({
    'PRESET': PRESET,
    'STAGE': STAGE,
    'SEEDS': SEEDS,
    'OUT_ROOT': OUT_ROOT,
    'DEVICE': DEVICE,
    'SKIP_COMPLETED': SKIP_COMPLETED,
})

{'PRESET': 'debug', 'STAGE': 'all', 'SEEDS': '1234', 'OUT_ROOT': 'runs/synthetic_niah_v4', 'DEVICE': 'cuda', 'SKIP_COMPLETED': True}


In [10]:
cmd = [
    sys.executable, '-u', '-m', 'synthetic_niah_v4.run_v4',
    '--preset', PRESET,
    '--stage', STAGE,
    '--device', DEVICE,
    '--out-root', OUT_ROOT,
    '--seeds', SEEDS,
]
if RUN_NAME:
    cmd += ['--run-name', RUN_NAME]
if SKIP_COMPLETED:
    cmd.append('--skip-completed')
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for line in proc.stdout:
    print(line, end='')
    captured.append(line.rstrip())
returncode = proc.wait()
if returncode:
    print('---- Last 120 log lines ----')
    print('\n'.join(captured[-120:]))
    raise subprocess.CalledProcessError(returncode, cmd)

RUN_DIR = None
for line in reversed(captured):
    if line.startswith('FINAL_RUN_DIR '):
        RUN_DIR = Path(line.split(' ', 1)[1].strip())
        break
if RUN_DIR is None:
    RUN_DIR = Path(OUT_ROOT) / RUN_NAME if RUN_NAME else Path(OUT_ROOT)
print('RUN_DIR =', RUN_DIR)

/usr/bin/python3 -u -m synthetic_niah_v4.run_v4 --preset debug --stage all --device cuda --out-root runs/synthetic_niah_v4 --seeds 1234 --skip-completed
[v4] stage=train
[v4 train] non_thinking seed=1234
[v4 train] thinking seed=1234
[v4] stage=behavior_eval
[v4] stage=cache
[v4] stage=probe
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.

## Result tables

The most important tables are behavior accuracy, probe results, direction diagnostics, and steering/patching. Probe accuracy alone means decodability; steering or patching is the causal check.

In [12]:
import pandas as pd
from IPython.display import Markdown, display

tables = RUN_DIR / 'tables'
for name in [
    'behavior_accuracy_by_count.csv',
    'probe_results.csv',
    'direction_metrics.csv',
    'steering_results.csv',
    'interchange_patching_results.csv',
]:
    path = tables / name
    display(Markdown(f'### `{name}`'))
    if path.exists() and path.stat().st_size > 0:
        display(pd.read_csv(path).head(20))
    else:
        display(Markdown('_No rows._'))

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Figures

These figures summarize: probe accuracy/R2 by anchor and layer, direction agreement with the count-token unembedding, steering dose response, steering controls, and patching from donor to receiver counts.

In [ ]:
from IPython.display import Image

for fig in sorted((RUN_DIR / 'figures').glob('*.png')):
    display(Markdown(f'### `{fig.name}`'))
    display(Image(filename=str(fig), width=900))

## Save to Google Drive

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_SAVE_COMPLETED = False
DRIVE_DIR = '/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results/'

if SAVE_TO_DRIVE and Path('/content').exists():
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    dest = Path(DRIVE_DIR) / RUN_DIR.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(RUN_DIR, dest)
    DRIVE_SAVE_COMPLETED = True
    print('saved to', dest)
else:
    print('SAVE_TO_DRIVE is False; local RUN_DIR:', RUN_DIR)

## Auto-disconnect Colab Runtime

This cell runs immediately after the Google Drive save cell. It only disconnects when a Drive save was confirmed; local VSCode/Jupyter runs are not force-closed by default.

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
FORCE_LOCAL_KERNEL_SHUTDOWN = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and globals().get("DRIVE_SAVE_COMPLETED", False):
    import time

    print("Google Drive save completed. Flushing Drive and disconnecting Colab runtime in 3 seconds...")
    time.sleep(3)
    try:
        from google.colab import drive, runtime

        try:
            drive.flush_and_unmount()
            print("Google Drive flushed and unmounted.")
        except Exception as e:
            print(f"Drive flush/unmount skipped or failed: {e}")
        runtime.unassign()
    except Exception as e:
        print(f"Colab runtime disconnect unavailable or failed: {e}")
        if FORCE_LOCAL_KERNEL_SHUTDOWN:
            import IPython

            IPython.Application.instance().kernel.do_shutdown(restart=False)
        else:
            print("Not forcing local kernel shutdown.")
else:
    print("Auto-disconnect skipped: no confirmed Google Drive save, or AUTO_DISCONNECT_AFTER_DRIVE_SAVE is False.")

## Optional GitHub result upload

In [ ]:
PUSH_RESULTS_TO_GITHUB = False
GIT_BRANCH = 'v4-results'

if PUSH_RESULTS_TO_GITHUB:
    subprocess.run(['git', 'checkout', '-B', GIT_BRANCH], check=True)
    subprocess.run(['git', 'add', str(RUN_DIR)], check=True)
    subprocess.run(['git', 'commit', '-m', f'Add v4 results {RUN_DIR.name}'], check=False)
    subprocess.run(['git', 'push', '-u', 'origin', GIT_BRANCH], check=True)
else:
    print('PUSH_RESULTS_TO_GITHUB is False')